In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter1d

# =======================
# 1. Generate Synthetic Heartbeat Data (Full-Day Simulation)
# =======================
np.random.seed(42)
minutes_per_day = 24 * 60
time = np.linspace(0, 24, minutes_per_day)  # in hours
hr = np.zeros_like(time)

# Morning exercise (5:30–6:30 AM)
exercise_mask = (time >= 5.5) & (time < 6.5)
hr[exercise_mask] = 130 + 10 * np.sin(np.linspace(0, 2*np.pi, exercise_mask.sum()))

# Commute (8:00–9:00 AM and 6:00–7:00 PM)
commute_mask1 = (time >= 8.0) & (time < 9.0)
commute_mask2 = (time >= 18.0) & (time < 19.0)
hr[commute_mask1] = 90 + 5 * np.sin(np.linspace(0, 2*np.pi, commute_mask1.sum()))
hr[commute_mask2] = 95 + 5 * np.sin(np.linspace(0, 2*np.pi, commute_mask2.sum()))

# Office work (9:00 AM–6:00 PM)
office_mask = (time >= 9.0) & (time < 18.0)
hr[office_mask] = 75 + 2 * np.sin(np.linspace(0, 6*np.pi, office_mask.sum()))

# Sleep (11:00 PM–5:30 AM)
sleep_mask = (time >= 23.0) | (time < 5.5)
hr[sleep_mask] = 60 + 1.5 * np.sin(np.linspace(0, 3*np.pi, sleep_mask.sum()))

# Remaining = resting
rest_mask = (hr == 0)
hr[rest_mask] = 70 + np.random.normal(0, 1, rest_mask.sum())

# Add noise and simulate occasional random spikes as abnormal events
hr += np.random.normal(0, 1.2, len(hr))
abnormal_indices = np.random.choice(len(hr), size=10, replace=False)
hr[abnormal_indices] += np.random.choice([15, -20], size=10)

# =======================
# 2. Predictive Coding Functions
# =======================
def predict_baseline(signal, smooth_sigma=30):
    return gaussian_filter1d(signal, sigma=smooth_sigma)

def compute_prediction_error(actual, predicted):
    return actual - predicted

def detect_layered_abnormalities(error, thresholds=(3, 5, 8)):
    """Detect anomalies in three layers of severity"""
    mild = np.abs(error) > thresholds[0]
    moderate = np.abs(error) > thresholds[1]
    severe = np.abs(error) > thresholds[2]
    return mild, moderate, severe

def refine_prediction(predicted, error, learning_rate=0.1, steps=10):
    for _ in range(steps):
        predicted += learning_rate * error
        error = compute_prediction_error(hr, predicted)
    return predicted

# =======================
# 3. Apply Predictive Coding
# =======================
predicted = predict_baseline(hr)
error = compute_prediction_error(hr, predicted)
mild, moderate, severe = detect_layered_abnormalities(error)
refined = refine_prediction(predicted.copy(), error)

# =======================
# 4. Plotting with Plotly
# =======================
fig = go.Figure()

# Add traces
fig.add_trace(go.Scatter(x=time, y=hr, mode='lines', name='Actual Heart Rate', 
                        line=dict(color='blue', width=1), opacity=0.6))
fig.add_trace(go.Scatter(x=time, y=predicted, mode='lines', name='Initial Prediction',
                        line=dict(color='green', width=1, dash='dash')))
fig.add_trace(go.Scatter(x=time, y=refined, mode='lines', name='Refined Prediction',
                        line=dict(color='red', width=2)))

# Add scatter points for abnormalities
fig.add_trace(go.Scatter(x=time[mild], y=hr[mild], mode='markers', name='Mild Abnormal',
                        marker=dict(color='orange', size=2, opacity=0.8)))
fig.add_trace(go.Scatter(x=time[moderate], y=hr[moderate], mode='markers', name='Moderate Abnormal',
                        marker=dict(color='red', size=4, opacity=0.8)))
fig.add_trace(go.Scatter(x=time[severe], y=hr[severe], mode='markers', name='Severe Abnormal',
                        marker=dict(color='purple', size=8, opacity=0.8)))

# Update layout
fig.update_layout(
    title='Daily Heartbeat Simulation with Predictive Coding & Multi-Layer Abnormal Detection',
    xaxis_title='Time (Hours)',
    yaxis_title='Heart Rate (bpm)',
    template='plotly_white',
    width=1500,
    height=600,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# Show the plot
fig.show()


In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter1d

# =======================
# 1. Generate Synthetic Heartbeat Data (Full-Day Simulation)
# =======================
np.random.seed(42)
minutes_per_day = 24 * 60
time = np.linspace(0, 24, minutes_per_day)  # in hours
hr = np.zeros_like(time)

# Morning exercise (5:30–6:30 AM)
exercise_mask = (time >= 5.5) & (time < 6.5)
hr[exercise_mask] = 130 + 10 * np.sin(np.linspace(0, 2*np.pi, exercise_mask.sum()))

# Commute (8:00–9:00 AM and 6:00–7:00 PM)
commute_mask1 = (time >= 8.0) & (time < 9.0)
commute_mask2 = (time >= 18.0) & (time < 19.0)
hr[commute_mask1] = 90 + 5 * np.sin(np.linspace(0, 2*np.pi, commute_mask1.sum()))
hr[commute_mask2] = 95 + 5 * np.sin(np.linspace(0, 2*np.pi, commute_mask2.sum()))

# Office work (9:00 AM–6:00 PM)
office_mask = (time >= 9.0) & (time < 18.0)
hr[office_mask] = 75 + 2 * np.sin(np.linspace(0, 6*np.pi, office_mask.sum()))

# Sleep (11:00 PM–5:30 AM)
sleep_mask = (time >= 23.0) | (time < 5.5)
hr[sleep_mask] = 60 + 1.5 * np.sin(np.linspace(0, 3*np.pi, sleep_mask.sum()))

# Remaining = resting
rest_mask = (hr == 0)
hr[rest_mask] = 70 + np.random.normal(0, 1, rest_mask.sum())

# Add noise and simulate occasional random spikes as abnormal events
hr += np.random.normal(0, 1.2, len(hr))
abnormal_indices = np.random.choice(len(hr), size=10, replace=False)
hr[abnormal_indices] += np.random.choice([15, -20], size=10)

# =======================
# 2. Predictive Coding Configuration
# =======================
# Define hierarchical layers with smoothing, threshold, and precision
layers = [
    {"name": "Layer 3 (Coarse)", "sigma": 60, "threshold": 8},
    {"name": "Layer 2 (Medium)", "sigma": 30, "threshold": 5},
    {"name": "Layer 1 (Fine)",   "sigma": 10, "threshold": 3},
]

# Initialize predictions and errors
for layer in layers:
    layer["pred"] = gaussian_filter1d(hr, sigma=layer["sigma"])
    layer["error"] = hr - layer["pred"]
    # Estimate precision = 1 / variance of error
    layer["precision"] = 1.0 / (np.var(layer["error"]) + 1e-6)

# Active inference baseline (initial copy of hr)
hr_estimate = hr.copy()

# Learning rate for refining
lr = 0.05

# =======================
# 3. One pass Predictive Coding Update
# =======================
for layer in layers:
    # Event-driven update: only where |error| > threshold
    mask = np.abs(layer["error"]) > layer["threshold"]
    
    # Precision-weighted correction
    correction = lr * layer["precision"] * layer["error"] * mask
    
    # Update prediction
    layer["pred"][mask] += correction[mask]
    
    # Recompute error after update
    layer["error"] = hr_estimate - layer["pred"]
    
    # Feedback: set estimate for next finer layer or final
    hr_estimate = layer["pred"]

# Active Inference: adjust observed hr_estimate closer to final prediction
action_gain = 0.1
action = action_gain * layers[-1]["error"]  # use top error to act
hr_corrected = hr - action  # simulate environment adjustment

# =======================
# 4. Plotting
# =======================
fig = go.Figure()

# Actual vs corrected
fig.add_trace(go.Scatter(x=time, y=hr, mode='lines', name='Actual HR',
                         line=dict(width=1), opacity=0.5))
fig.add_trace(go.Scatter(x=time, y=hr_corrected, mode='lines', name='After Action (Active Inference)',
                         line=dict(width=2, dash='dot')))

# Predictions at each layer
for layer in layers:
    fig.add_trace(go.Scatter(x=time, y=layer["pred"], mode='lines', name=layer["name"]))

# Mark abnormal events detected at fine layer
abnormals = np.abs(layers[2]["error"]) > layers[2]["threshold"]
fig.add_trace(go.Scatter(x=time[abnormals], y=hr[abnormals], mode='markers',
                         name='Detected Abnormal (Layer 1)', marker=dict(size=6, color='red')))

fig.update_layout(
    title='Hierarchical & Event-Driven Predictive Coding with Precision & Active Inference',
    xaxis_title='Time (Hours)', yaxis_title='Heart Rate (bpm)',
    template='plotly_white', width=1400, height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.update_xaxes(showgrid=True, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridcolor='LightGray')

fig.show()
